# Leereenheid 4.3: Kenmerkingenieurswese

## Skep nuwe kenmerke uit bestaande data om modelprestasie te verbeter en evalueer die impak van hierdie kenmerke

### Gevallestudie: Boland Meubels & Toestelle

Pieter wil nuwe metings skep wat hom help om sy winsmargeprobleem beter te verstaan.

In [ ]:
# Laai nodige biblioteke
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Stap 1: Laai die getransformeerde data

Ons begin met die getransformeerde data van LU 4.2.

In [ ]:
# Laai die getransformeerde verkoopsdata
url = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_verkope_getransformeer.csv'
data = pd.read_csv(url)

# Skakel datum na datetime
data['datum'] = pd.to_datetime(data['datum'])

# Kyk na die eerste paar rye
data.head()

## Stap 2: Wiskundige transformasies

Ons skep eenvoudige finansiële kenmerke uit bestaande data.

In [ ]:
# Bereken winsmargin as persentasie
data['winsmarge'] = ((data['verkoopsprys'] - data['kosprys']) / data['verkoopsprys']) * 100

# Bereken bruto wins in Rand
data['bruto_wins'] = data['verkoopsprys'] - data['kosprys']

# Bereken totale inkomste per transaksie (indien nie reeds bestaan nie)
if 'totale_inkomste' not in data.columns:
    data['totale_inkomste'] = data['verkoopsprys'] * data['hoeveelheid']

# Wys die nuwe kenmerke
print("Nuwe finansiële kenmerke:")
data[['verkoopsprys', 'kosprys', 'winsmarge', 'bruto_wins', 'totale_inkomste']].head()

In [ ]:
# Kyk na beskrywende statistiek van winsmarge
print("Winsmarge statistiek:")
print(data['winsmarge'].describe())

# Visualiseer winsmarge verspreiding
plt.figure(figsize=(10, 5))
plt.hist(data['winsmarge'], bins=30, edgecolor='black')
plt.xlabel('Winsmarge (%)')
plt.ylabel('Frekwensie')
plt.title('Verspreiding van Winsmargin')
plt.axvline(data['winsmarge'].mean(), color='red', linestyle='--', label=f"Gemiddeld: {data['winsmarge'].mean():.1f}%")
plt.legend()
plt.show()

## Stap 3: Tydsgebaseerde kenmerke

Ons skep kenmerke wat seisoenaliteit en tydspatrone onthul.

In [ ]:
# Onttrek addisionele tydskomponente (indien nie reeds bestaan nie)
if 'jaar' not in data.columns:
    data['jaar'] = data['datum'].dt.year
if 'maand' not in data.columns:
    data['maand'] = data['datum'].dt.month
if 'kwartaal' not in data.columns:
    data['kwartaal'] = data['datum'].dt.quarter

# Skep maand naam
data['maand_naam'] = data['datum'].dt.month_name()

# Skep is_naweek kenmerk
data['is_naweek'] = data['datum'].dt.dayofweek >= 5

# Skep seisoen kenmerk (Suidelike Halfrond)
seisoen_map = {
    12: 'Somer', 1: 'Somer', 2: 'Somer',
    3: 'Herfs', 4: 'Herfs', 5: 'Herfs',
    6: 'Winter', 7: 'Winter', 8: 'Winter',
    9: 'Lente', 10: 'Lente', 11: 'Lente'
}
data['seisoen'] = data['maand'].map(seisoen_map)

# Wys voorbeelde
print("Tydsgebaseerde kenmerke:")
data[['datum', 'jaar', 'maand', 'kwartaal', 'is_naweek', 'seisoen']].head(10)

In [ ]:
# Analiseer verkope per seisoen
verkope_per_seisoen = data.groupby('seisoen')['totale_inkomste'].agg(['sum', 'mean', 'count'])
print("\nVerkope per seisoen:")
print(verkope_per_seisoen)

## Stap 4: Kategorisering (*Binning*)

Ons skep kategorieë uit kontinue veranderlikes.

In [ ]:
# Skep verkope kategorie (Laag, Medium, Hoog)
data['verkope_kategorie'] = pd.cut(
    data['totale_inkomste'],
    bins=[0, 5000, 15000, float('inf')],
    labels=['Laag', 'Medium', 'Hoog']
)

# Skep winsmargin kategorie (Swak, Gemiddeld, Goed)
data['winsmarge_kategorie'] = pd.cut(
    data['winsmarge'],
    bins=[0, 25, 30, 100],
    labels=['Swak', 'Gemiddeld', 'Goed']
)

# Wys die resultaat
print("Kategorisering voorbeelde:")
data[['totale_inkomste', 'verkope_kategorie', 'winsmarge', 'winsmarge_kategorie']].head(10)

In [ ]:
# Tel transaksies per winsmarge kategorie
print("\nVerspreiding van winsmarge kategorieë:")
print(data['winsmarge_kategorie'].value_counts())

## Stap 5: Samevatting per groep

Ons skep opsommende statistiek per winkel, produk, en kliënt.

In [ ]:
# Totale verkope per winkel
verkope_per_winkel = data.groupby('winkel_naam').agg({
    'totale_inkomste': ['sum', 'mean', 'count'],
    'winsmarge': 'mean'
}).round(2)

print("Verkope per winkel:")
print(verkope_per_winkel)

In [ ]:
# Gemiddelde winsmarge per produk
winsmarge_per_produk = data.groupby('produk')['winsmarge'].agg(['mean', 'std', 'count']).round(2)
winsmarge_per_produk = winsmarge_per_produk.sort_values('mean', ascending=False)

print("\nGemiddelde winsmarge per produk:")
print(winsmarge_per_produk)

In [ ]:
# Aantal transaksies per kliënt (Frequency in RFM)
transaksies_per_klient = data.groupby('klient_id').size()

print("\nBeskrywende statistiek vir transaksies per kliënt:")
print(transaksies_per_klient.describe())

## Stap 6: Domein-spesifieke kenmerke

Ons skep rekeningkundige kenmerke wat relevant is vir finansiële analise.

In [ ]:
# Bereken wins opslag persentasie (gebaseer op kosprys)
data['wins_opslag_pct'] = ((data['verkoopsprys'] - data['kosprys']) / data['kosprys']) * 100

# Bereken bydrae marge (aanvaar gemiddelde vaste koste van R50 per eenheid)
vaste_koste_per_eenheid = 50
data['bydrae_marge'] = data['bruto_wins'] - vaste_koste_per_eenheid

# Wys die nuwe kenmerke
print("Domein-spesifieke kenmerke:")
data[['verkoopsprys', 'kosprys', 'winsmarge', 'wins_opslag_pct', 'bydrae_marge']].head()

In [ ]:
# Identifiseer produkte met negatiewe bydrae marge
negatiewe_bydrae = data[data['bydrae_marge'] < 0]
print(f"\nAantal transaksies met negatiewe bydrae marge: {len(negatiewe_bydrae)}")

if len(negatiewe_bydrae) > 0:
    print("\nProdukte met negatiewe bydrae (verlies maak):")
    print(negatiewe_bydrae.groupby('produk')['bydrae_marge'].agg(['count', 'mean']).sort_values('count', ascending=False))

## Stap 7: Winsmargin tendensanalise

Kom ons ondersoek of winsmarges werklik daal oor tyd.

In [ ]:
# Bereken gemiddelde winsmargin per maand
data['jaar_maand'] = data['datum'].dt.to_period('M')
winsmarge_per_maand = data.groupby('jaar_maand')['winsmarge'].mean()

# Visualiseer die tendens
plt.figure(figsize=(12, 6))
winsmarge_per_maand.plot(marker='o')
plt.xlabel('Maand')
plt.ylabel('Gemiddelde Winsmarge (%)')
plt.title('Winsmarge Tendens oor Tyd')
plt.axhline(y=35, color='g', linestyle='--', label='Teiken: 35%')
plt.axhline(y=28, color='r', linestyle='--', label='Huidige: 28%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nGemiddelde winsmarge in eerste maand: {winsmarge_per_maand.iloc[0]:.2f}%")
print(f"Gemiddelde winsmarge in laaste maand: {winsmarge_per_maand.iloc[-1]:.2f}%")
print(f"Afname: {winsmarge_per_maand.iloc[0] - winsmarge_per_maand.iloc[-1]:.2f}%")

## Stap 8: Finale datastel met alle kenmerke

Kom ons kyk na al die nuwe kenmerke wat ons geskep het.

In [ ]:
# Lys van alle nuwe kenmerke
nuwe_kenmerke = [
    'winsmarge',
    'bruto_wins',
    'totale_inkomste',
    'maand_naam',
    'is_naweek',
    'seisoen',
    'verkope_kategorie',
    'winsmarge_kategorie',
    'wins_opslag_pct',
    'bydrae_marge'
]

print("Nuwe kenmerke wat geskep is:")
for kenmerk in nuwe_kenmerke:
    print(f"  - {kenmerk}")

print(f"\nTotale aantal kolomme: {len(data.columns)}")

In [ ]:
# Wys finale data
data.head()

## Stap 9: Stoor die verrykte data

Stoor die data met al die nuwe kenmerke vir gebruik in LU 4.4.

In [ ]:
# Stoor die verrykte data
data.to_csv('boland_meubels_verkope_verryk.csv', index=False)
print("Verrykte data gestoor as: boland_meubels_verkope_verryk.csv")

## Opsomming

In hierdie notebook het ons:
1. Wiskundige transformasies toegepas om finansiële kenmerke te skep (winsmargin, bruto wins, totale inkomste)
2. Tydsgebaseerde kenmerke geskep (seisoen, is naweek, maand naam)
3. Kategorisering toegepas op verkope en winsmargin
4. Aggregasie gebruik om opsommings per winkel, produk, en kliënt te skep
5. Domein-spesifieke rekeningkundige kenmerke geskep (markup persentasie, bydrae marge)
6. Winsmargin tendense ontleed om die dalende patroon te bevestig

Hierdie nuwe kenmerke help Pieter om:
- Watter produkte swak presteer te identifiseer
- Seisoenale patrone te herken
- Winkel prestasie te vergelyk
- Verlieslatende produkte te identifiseer

Die data is nou gereed vir data-integrasie in LU 4.4.